# Delta Hedge Comparison

## Libraries Import

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')

from src.config_loader import load_config
from src.stochastic_engines import GBMModel, MertonJumpDiffusion
from src.analytics import (
    bsm_price,
    bsm_delta,
)

np.set_printoptions(suppress=True, precision=6)

## Market Configuration

In [ ]:
config = load_config()

config.market = config.market_params
config.merton = config.merton_params
config.sim = config.simulation

r = float(config.market.r)
sigma_bsm = float(config.market.sigma)
dt = float(config.market.dt)
S0 = float(config.market.S0)

hedge_T = 10.0  # Hedge horizon
n_steps = int(round(hedge_T / dt))
t_grid = np.arange(n_steps + 1) * dt

def _truncate_paths(paths: np.ndarray, n_steps: int) -> np.ndarray:
    return np.asarray(paths[:, : n_steps + 1], dtype=float)

print(f"dt={dt:.6f}, hedge_T={hedge_T}, n_steps={n_steps}, n_paths={config.sim.n_paths}")

## Loading Simulation

In [ ]:
from pathlib import Path

# Paths access
paths_file = Path("..") / "data" / "simulated_paths.npz"

if paths_file.exists():
    data = np.load(paths_file)
    gbm_paths_full = np.asarray(data["gbm"], dtype=float)
    mjd_paths_full = np.asarray(data["merton"], dtype=float)

    # Error if dt mismatch
    dt_file = float(data.get("dt", dt))
    if not np.isclose(dt_file, dt):
        print(f"Warning: dt in file ({dt_file}) != config dt ({dt}).")

    print("Loaded simulated paths...")
else:
    print("Paths file not found...")
    gbm_paths_full = GBMModel(config).generate_paths()
    mjd_paths_full = MertonJumpDiffusion(config).generate_paths()

# Truncation to hedge horizon
gbm_paths = _truncate_paths(gbm_paths_full, n_steps=n_steps)
mjd_paths = _truncate_paths(mjd_paths_full, n_steps=n_steps)

assert gbm_paths.shape == mjd_paths.shape
print("paths shape:", gbm_paths.shape)

## Volatility Skew

In [ ]:
from scipy.optimize import brentq

T_smile = hedge_T
S0_smile = S0
r_smile = r

K_grid = np.linspace(0.5 * S0_smile, 1.5 * S0_smile, 15)
ST_mjd = mjd_paths[:, -1]
discount = np.exp(-r_smile * T_smile)

def implied_vol(price, S, K, T, r, option_type="call", tol=1e-6):
    if price <= 0.0:
        return np.nan

    def _f(sigma):
        return bsm_price(S, K, T, r, sigma, option_type=option_type) - price

    try:
        return brentq(_f, 1e-4, 5.0, xtol=tol)
    except ValueError:
        return np.nan

def skew_from_sample(ST_sample):
    prices = []
    for K in K_grid:
        payoff = np.maximum(ST_sample - K, 0.0)
        prices.append(discount * np.mean(payoff))
    prices = np.asarray(prices, dtype=float)
    iv = np.array([
        implied_vol(p, S0_smile, K, T_smile, r_smile)
        for p, K in zip(prices, K_grid)
    ])
    return iv

# Skew from terminal distribution
rng = np.random.default_rng(123)
n_paths = ST_mjd.shape[0]
n_boot = 50
boot_iv = []

plt.figure(figsize=(9, 5))

for b in range(n_boot):
    idx = rng.choice(n_paths, size=n_paths, replace=True)
    iv = skew_from_sample(ST_mjd[idx])
    boot_iv.append(iv)
    plt.plot(K_grid / S0_smile, iv, color="purple", alpha=0.15, lw=1)

boot_iv = np.asarray(boot_iv)
iv_med = np.nanmedian(boot_iv, axis=0)
iv_p05 = np.nanpercentile(boot_iv, 5, axis=0)
iv_p95 = np.nanpercentile(boot_iv, 95, axis=0)

plt.plot(K_grid / S0_smile, iv_med, color="purple", lw=2.5, label="MJD")
plt.fill_between(K_grid / S0_smile, iv_p05, iv_p95, color="purple", alpha=0.15, label="5–95%")

plt.axhline(sigma_bsm, linestyle="--", label="GBM", color="green")
plt.xlabel("Moneyness")
plt.ylabel("Implied volatility")
plt.title("Volatility Skew")
plt.legend()
plt.tight_layout()
plt.show()

## Hedging Contract

In [ ]:
K0 = S0  # ATM strike
option_type = "call"

## Delta Hedge Engine

In [ ]:
def time_to_maturity_grid(T: float, dt: float, n_steps: int) -> np.ndarray:
    tau = T - np.arange(n_steps + 1) * dt
    return np.maximum(tau, 0.0)


tau_grid = time_to_maturity_grid(hedge_T, dt=dt, n_steps=n_steps)


def run_delta_hedge(
    paths: np.ndarray,
    *,
    r: float,
    sigma: float,
    dt: float,
    tau_grid: np.ndarray,
    K0: float,
    option_type: str = "call",
) -> dict:
    paths = np.asarray(paths, dtype=float)
    n_paths, n_steps_plus_1 = paths.shape
    n_steps = n_steps_plus_1 - 1

    assert tau_grid.shape[0] == n_steps_plus_1

    # Holdings
    xS = np.zeros((n_paths, n_steps_plus_1))  # Stock
    B = np.zeros((n_paths, n_steps_plus_1))   # Bank account

    # Liability model price
    C0 = np.zeros((n_paths, n_steps_plus_1))

    # Pre-rebalance delta exposure
    net_delta_pre = np.full((n_paths, n_steps_plus_1), np.nan)

    # Stepwise PnL components
    pnl_stock = np.zeros((n_paths, n_steps))
    pnl_cash = np.zeros((n_paths, n_steps))
    pnl_liability = np.zeros((n_paths, n_steps))
    pnl_hedged = np.zeros((n_paths, n_steps))

    # Initial hedge
    S0_vec = paths[:, 0]
    tau0 = float(tau_grid[0])

    C0[:, 0] = bsm_price(S0_vec, K0, tau0, r, sigma, option_type=option_type)
    D0 = bsm_delta(S0_vec, K0, tau0, r, sigma, option_type=option_type)

    xS[:, 0] = D0
    B[:, 0] = C0[:, 0] - xS[:, 0] * S0_vec

    growth = float(np.exp(r * dt))

    for i in range(n_steps):
        S_i = paths[:, i]
        S_ip1 = paths[:, i + 1]
        tau_i = float(tau_grid[i])
        tau_ip1 = float(tau_grid[i + 1])

        # Current liability price
        if i > 0:
            C0[:, i] = bsm_price(S_i, K0, tau_i, r, sigma, option_type=option_type)

        # Next liability price and delta
        C0_ip1 = bsm_price(S_ip1, K0, tau_ip1, r, sigma, option_type=option_type)
        D0_ip1 = bsm_delta(S_ip1, K0, tau_ip1, r, sigma, option_type=option_type)

        # PnL components
        pnl_stock[:, i] = xS[:, i] * (S_ip1 - S_i)
        pnl_cash[:, i] = B[:, i] * (growth - 1.0)
        pnl_liability[:, i] = (C0_ip1 - C0[:, i])

        pnl_hedged[:, i] = pnl_stock[:, i] + pnl_cash[:, i] - pnl_liability[:, i]

        # Portfolio before re-hedging
        V_pre = xS[:, i] * S_ip1 + B[:, i] * growth

        # Pre-rebalance delta exposure
        net_delta_pre[:, i + 1] = xS[:, i] - D0_ip1

        # Re-hedge portfolio position
        xS[:, i + 1] = D0_ip1
        C0[:, i + 1] = C0_ip1
        B[:, i + 1] = V_pre - xS[:, i + 1] * S_ip1

    # Terminal portfolio value and hedge errors
    V = xS * paths + B
    hedge_error = V - C0

    payoff = (
        np.maximum(paths[:, -1] - K0, 0.0)
        if option_type == "call"
        else np.maximum(K0 - paths[:, -1], 0.0)
    )
    terminal_error_vs_payoff = V[:, -1] - payoff

    return {
        "paths": paths,
        "V": V,
        "C0": C0,
        "xS": xS,
        "B": B,
        "hedge_error": hedge_error,
        "terminal_error_vs_payoff": terminal_error_vs_payoff,
        "pnl_stock": pnl_stock,
        "pnl_cash": pnl_cash,
        "pnl_liability": pnl_liability,
        "pnl_hedged": pnl_hedged,
        "net_delta_pre": net_delta_pre,
    }


def summarize_terminal_error(err_T: np.ndarray) -> pd.Series:
    err_T = np.asarray(err_T, dtype=float)
    return pd.Series({
        "mean": float(np.mean(err_T)),
        "std": float(np.std(err_T, ddof=1)),
        "rmse": float(np.sqrt(np.mean(err_T**2))),
        "q01": float(np.quantile(err_T, 0.01)),
        "q05": float(np.quantile(err_T, 0.05)),
        "q50": float(np.quantile(err_T, 0.50)),
        "q95": float(np.quantile(err_T, 0.95)),
        "q99": float(np.quantile(err_T, 0.99)),
    })

## Running Hedge

In [ ]:
gbm_res = run_delta_hedge(
    gbm_paths,
    r=r,
    sigma=sigma_bsm,
    dt=dt,
    tau_grid=tau_grid,
    K0=K0,
    option_type=option_type,
)

mjd_res = run_delta_hedge(
    mjd_paths,
    r=r,
    sigma=sigma_bsm,
    dt=dt,
    tau_grid=tau_grid,
    K0=K0,
    option_type=option_type,
)

gbm_term = gbm_res["terminal_error_vs_payoff"]
mjd_term = mjd_res["terminal_error_vs_payoff"]

summary = pd.DataFrame({
    "GBM": summarize_terminal_error(gbm_term),
    "MertonJD": summarize_terminal_error(mjd_term),
})

summary

## Hedge error (time series + terminal distribution)

We report hedge error both as:
- **mark-to-model**: \(E_t = V_t - C_0(S_t,\tau_t)\)
- **terminal vs payoff**: \(V_T - \text{payoff}\) (this is the economically relevant replication error)

In [ ]:
def plot_error_timeseries(res: dict, title: str, *, t_grid: np.ndarray, color: str):
    E = res["hedge_error"]
    q = np.quantile(E, [0.05, 0.5, 0.95], axis=0)
    mean = E.mean(axis=0)

    plt.figure(figsize=(11, 4))
    plt.plot(t_grid, mean, lw=2.0, label="mean", color=color)
    plt.plot(t_grid, q[1], lw=2.0, label="median", color=color)
    plt.fill_between(t_grid, q[0], q[2], alpha=0.2, label="5–95%", color=color)
    plt.title(title)
    plt.xlabel("time (years)")
    plt.ylabel("E(t)=V(t)-C0(t)")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_terminal_distribution(err_T: np.ndarray, title: str, *, color: str):
    err_T = np.asarray(err_T, dtype=float)
    plt.figure(figsize=(10, 4))
    sns.histplot(err_T, bins=70, stat="density", color=color, alpha=0.35)
    sns.kdeplot(err_T, color=color, lw=2)
    plt.title(title)
    plt.xlabel("terminal hedge error")
    plt.tight_layout()
    plt.show()


plot_error_timeseries(gbm_res, "GBM: mark-to-model hedge error E(t)", t_grid=t_grid, color="green")
plot_error_timeseries(mjd_res, "Merton jump-diffusion: mark-to-model hedge error E(t)", t_grid=t_grid, color="purple")

plot_terminal_distribution(gbm_term, "GBM: terminal replication error V_T - payoff", color="green")
plot_terminal_distribution(mjd_term, "Merton jump-diffusion: terminal replication error V_T - payoff", color="purple")

In [ ]:
def plot_pnl_distribution(res: dict, title: str, *, color: str):
    pnl_T = res["pnl_hedged"].sum(axis=1)
    plt.figure(figsize=(10, 4))
    sns.histplot(pnl_T, bins=70, stat="density", color=color, alpha=0.35)
    sns.kdeplot(pnl_T, color=color, lw=2)
    plt.axvline(0.0, color="black", lw=1)
    plt.title(title)
    plt.xlabel("terminal hedged PnL")
    plt.tight_layout()
    plt.show()


def plot_attribution_stacked(res: dict, title: str, *, color: str, n_paths_plot: int = 200):
    rng = np.random.default_rng(123)
    n_paths = res["pnl_hedged"].shape[0]
    idx = rng.choice(n_paths, size=min(n_paths_plot, n_paths), replace=False)

    stock = res["pnl_stock"][idx].sum(axis=1)
    cash = res["pnl_cash"][idx].sum(axis=1)
    liab = -res["pnl_liability"][idx].sum(axis=1)  # minus liability PnL

    df = pd.DataFrame({
        "stock": stock,
        "cash": cash,
        "-liability": liab,
    })

    plt.figure(figsize=(11, 4.5))
    df_melt = df.melt(var_name="component", value_name="pnl")
    sns.violinplot(data=df_melt, x="component", y="pnl", inner="quartile", cut=0, color=color)
    plt.title(title)
    plt.axhline(0.0, color="black", lw=1)
    plt.tight_layout()
    plt.show()

plot_pnl_distribution(gbm_res, "GBM: terminal hedged PnL distribution", color="green")
plot_pnl_distribution(mjd_res, "Merton jump-diffusion: terminal hedged PnL distribution", color="purple")

plot_attribution_stacked(gbm_res, "GBM: terminal PnL attribution (sample paths)", color="green")
plot_attribution_stacked(mjd_res, "Merton jump-diffusion: terminal PnL attribution (sample paths)", color="purple")

## Portfolio value paths

We plot sample paths of the **replicating portfolio value** \(V_t\) vs the model price \(C_0(S_t,\tau_t)\).

In [ ]:
def plot_portfolio_paths(res: dict, title: str, *, t_grid: np.ndarray, color: str, n_plot: int = 10, seed: int = 0):
    rng = np.random.default_rng(seed)
    n_paths = res["V"].shape[0]
    idx = rng.choice(n_paths, size=min(n_plot, n_paths), replace=False)

    V = res["V"][idx]
    C0 = res["C0"][idx]

    plt.figure(figsize=(12, 5))
    for j in range(V.shape[0]):
        plt.plot(t_grid, V[j], color=color, alpha=0.15)
    plt.plot(t_grid, np.median(V, axis=0), color=color, lw=2.2, label="V median")

    # show median model price too
    plt.plot(t_grid, np.median(C0, axis=0), color=color, lw=2.2, ls="--", label="C0 median")

    plt.title(title)
    plt.xlabel("time (years)")
    plt.ylabel("value")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_portfolio_paths(gbm_res, "GBM: replicating portfolio value V(t)", t_grid=t_grid, color="green", seed=1)
plot_portfolio_paths(mjd_res, "Merton jump-diffusion: replicating portfolio value V(t)", t_grid=t_grid, color="purple", seed=2)

## Delta Exposure Plots

In [ ]:
def plot_exposure(res: dict, title: str, *, t_grid: np.ndarray, color: str):
    d = res["net_delta_pre"]

    d_medabs = np.nanmedian(np.abs(d), axis=0)

    plt.figure(figsize=(7, 4))
    plt.plot(t_grid, d_medabs, lw=2, color=color)
    plt.title("median |net delta| (pre-rebalance)")
    plt.xlabel("time (years)")
    plt.ylabel("exposure")
    plt.tight_layout()
    plt.show()


plot_exposure(gbm_res, "GBM: pre-rebalance delta exposure", t_grid=t_grid, color="green")
plot_exposure(mjd_res, "Merton jump-diffusion: pre-rebalance delta exposure", t_grid=t_grid, color="purple")